# 01. GeoLife Data Exploration & Quality Verification

**Project**: AI-Driven Predictive Geofencing and Trajectory Analysis for Proactive Safety Monitoring  
**Stage**: Chunk 1 — Ingestion, Validation & Minimal Cleaning Exploration  

This notebook provides exploratory data analysis, quality validation, and statistical summaries for the canonical processed Microsoft GeoLife dataset saved in `ml/data/processed/trajectories.parquet`.

> **Note**: All cleaning, ingestion, and validation logic is centralized in [`ml/src/data.py`](../src/data.py). This notebook is strictly for exploration and verification.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

# Resolve paths dynamically without hardcoded absolute paths
NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "ml" / "notebooks"
PROCESSED_DIR = NOTEBOOK_DIR.parent / "data" / "processed"
PARQUET_FILE = PROCESSED_DIR / "trajectories.parquet"
REPORT_FILE = PROCESSED_DIR / "data_quality_report.json"

print(f"Processed Directory: {PROCESSED_DIR}")
print(f"Parquet Dataset:     {PARQUET_FILE} (exists: {PARQUET_FILE.exists()})")
print(f"Data Quality Report: {REPORT_FILE} (exists: {REPORT_FILE.exists()})")

## 1. Load Data Quality Report & Ingestion Diagnostics

In [ ]:
if REPORT_FILE.exists():
    with open(REPORT_FILE, "r", encoding="utf-8") as f:
        quality_report = json.load(f)
    print("--- Data Quality Ingestion Report ---")
    for k, v in quality_report.items():
        if isinstance(v, dict):
            print(f"\n{k.upper()}:")
            for sub_k, sub_v in v.items():
                print(f"  {sub_k}: {sub_v}")
        else:
            print(f"{k}: {v}")
else:
    print("Data quality report not found yet. Run `python ml/src/data.py` first.")

## 2. Load Processed Parquet Dataset

In [ ]:
# Load canonical parquet dataset
df = pd.read_parquet(PARQUET_FILE)
print(f"Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Memory Usage:  {df.memory_usage(deep=True).sum() / (1024 * 1024):.2f} MB")
df.info()

## 3. Dataset Head & Schema Inspection

In [ ]:
df.head(10)

## 4. User and Trajectory Counts

In [ ]:
num_users = df["user_id"].nunique()
num_trajectories = df["trajectory_id"].nunique()
print(f"Unique Users:        {num_users}")
print(f"Unique Trajectories: {num_trajectories:,}")

# Trajectories per user distribution
traj_per_user = df.groupby("user_id")["trajectory_id"].nunique()
print("\nTrajectories per User Summary:")
print(traj_per_user.describe())

## 5. Missing Values & Integrity Verification

In [ ]:
missing = df.isna().sum()
print("Missing values per column:")
print(missing)
assert missing["timestamp"] == 0, "Found missing timestamps!"
assert missing["latitude"] == 0, "Found missing latitudes!"
assert missing["longitude"] == 0, "Found missing longitudes!"
print("\nData integrity check PASSED: zero missing values in core coordinates and timestamps.")

## 6. Trajectory Duration Statistics

In [ ]:
# Duration per trajectory (seconds)
traj_spans = df.groupby("trajectory_id")["timestamp"].agg(["min", "max", "count"])
traj_spans["duration_sec"] = (traj_spans["max"] - traj_spans["min"]).dt.total_seconds()
traj_spans["duration_min"] = traj_spans["duration_sec"] / 60.0

print("Trajectory Points Count:")
print(traj_spans["count"].describe(percentiles=[0.25, 0.5, 0.75, 0.95]))

print("\nTrajectory Duration (seconds):")
print(traj_spans["duration_sec"].describe(percentiles=[0.25, 0.5, 0.75, 0.95]))

print("\nTrajectory Duration (minutes):")
print(traj_spans["duration_min"].describe(percentiles=[0.25, 0.5, 0.75, 0.95]))

## 7. GPS Sampling Interval Statistics

In [ ]:
# Calculate consecutive timestamp differences within trajectories
# (Sample first 500 trajectories for interactive display speed)
sample_traj_ids = df["trajectory_id"].drop_duplicates().head(500)
sample_df = df[df["trajectory_id"].isin(sample_traj_ids)].sort_values(["trajectory_id", "timestamp"])

intervals = sample_df.groupby("trajectory_id")["timestamp"].diff().dt.total_seconds().dropna()

print("Sampling Interval Statistics (seconds):")
print(intervals.describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]))

## 8. Basic Coordinate Distribution

In [ ]:
coord_stats = df[["latitude", "longitude", "altitude"]].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
)
print("Coordinate & Altitude Distributions:")
print(coord_stats)

# Check coordinate bounds
print(f"\nLatitude Range:  [{df['latitude'].min():.6f}, {df['latitude'].max():.6f}]")
print(f"Longitude Range: [{df['longitude'].min():.6f}, {df['longitude'].max():.6f}]")

## 9. Reproducible Example Trajectories

In [ ]:
example_ids = df["trajectory_id"].drop_duplicates().head(3).tolist()
for tid in example_ids:
    sub = df[df["trajectory_id"] == tid]
    duration_s = (sub["timestamp"].iloc[-1] - sub["timestamp"].iloc[0]).total_seconds()
    print(f"=== Trajectory: {tid} ===")
    print(f"User: {sub['user_id'].iloc[0]} | Points: {len(sub)} | Start: {sub['timestamp'].iloc[0]} | End: {sub['timestamp'].iloc[-1]} | Duration: {duration_s:.1f}s")
    print(sub[["timestamp", "latitude", "longitude", "altitude"]].head(3))
    print()
